<div style="background: linear-gradient(135deg, #080c14 0%, #111d31 50%, #1a0a2e 100%); padding: 40px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: #4f8ef7; font-size: 2.2em; margin: 0; font-family: 'Segoe UI', sans-serif;">🎙️ LiveAvatar Streaming Pipeline</h1>
  <h2 style="color: #8b5cf6; font-size: 1.2em; font-weight: 300; margin: 10px 0 0 0;">Moshi · Bridge (ONNX) · Ditto — Real-Time Talking Head</h2>
  <p style="color: #64748b; margin: 14px 0 0 0; font-size: 0.9em;">
    Browser Mic → Moshi Streaming → Bridge (ONNX / .pt) → Ditto Online Mode → Live Video + Audio
  </p>
</div>

---

### 📋 How this works

| Component | Role | Mode |
|-----------|------|------|
| **Moshi** | Real-time speech + voice generation | `streaming_forever` — raw PCM audio |
| **Bridge** | Mimi tokens → HuBERT features | **ONNX Runtime** (fastest) or `.pt` fallback |
| **Ditto** | Continuous talking-head animation | `online_mode=True` — JPEG frames |
| **FastAPI** | WebSocket server + browser UI | AudioWorklet PCM playback |

> **Requirements**: RunPod GPU ≥ A100 40 GB.  
> **Audio fix**: server now sends raw float32 PCM → browser AudioWorklet plays it directly (no Opus wrapping issues).  
> **Speed fix**: ONNX bridge processes each token immediately (chunk=1) instead of batching 4 tokens.

In [1]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/moshi-ditto-streaming-debug-v11.git

Cloning into 'moshi-ditto-streaming-debug-v11'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 158 (delta 7), reused 155 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.09 MiB | 11.08 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
%cd /workspace/moshi-ditto-streaming-debug-v11

/workspace/moshi-ditto-streaming-debug-v11


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


---
## 🔧 Cell 1 — Install Dependencies

Run **once** per RunPod session.

In [3]:
import os, subprocess, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

install_script = os.path.join(PROJECT_ROOT, "install.sh")
if not os.path.isfile(install_script):
    raise FileNotFoundError(f"install.sh not found at: {install_script}")

print("🚀 Starting installation...\n")
result = subprocess.run(["bash", install_script], cwd=PROJECT_ROOT, text=True)
if result.returncode != 0:
    print("❌ Installation failed — check output above.")
else:
    print("\n✅ Installation complete!")

📁 Project root: /workspace/moshi-ditto-streaming-debug-v11
🚀 Starting installation...


═══════════════════════════════════════════════════════════════
   Moshi + Bridge + Ditto — Unified Pipeline Installer
═══════════════════════════════════════════════════════════════

[install] Detected CUDA: 12.4
[install] Step 1/7 — Installing PyTorch 2.5.1+cu124 ...


[install] ✅  PyTorch installed.
[install] Step 2/7 — Installing Moshi from local source (moshi-inference/) ...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


[install] ✅  Moshi installed.
[install] Step 3/7 — Installing core audio/NLP packages ...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


...

[install] ⚠  TensorRT installation had warnings (may be OK if already installed).
[install] ✅  TensorRT setup done.
[install] Step 7/8 — Installing miscellaneous utilities ...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


[install] ✅  Miscellaneous utilities installed.
[install] Step 8/8 — Installing streaming server dependencies (FastAPI / uvicorn) ...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


[install] ✅  Streaming server dependencies installed.
[install] ⚠  FFmpeg not found in PATH.
[install]    Install with:  apt-get update && apt-get install -y ffmpeg
[install]    (Required for the final audio+video merge step)
[install] Checkpoint directory: /workspace/moshi-ditto-streaming-debug-v11/checkpoints
[install] Place your trained bridge model at: checkpoints/bridge_best.pt


═══════════════════════════════════════════════════════════════
   ✅  All dependencies installed successfully!

   Next steps:
   1. Ensure bridge checkpoint: checkpoints/bridge_best.pt
   2. Ensure Ditto TRT models in:
      ditto-inference/checkpoints/ditto_trt_Ampere_Plus/
   3. Open MoshiDittoPipeline.ipynb and run all cells
═══════════════════════════════════════════════════════════════


✅ Installation complete!


In [4]:
# System packages + cuDNN (needed for TRT and ONNX CUDA provider)
!apt update -qq
!apt-get install -y ffmpeg libcudnn8 libcudnn8-dev 2>&1 | tail -5

146 packages can be upgraded. Run 'apt list --upgradable' to see them.
Setting up libavfilter7:amd64 (7:4.4.2-0ubuntu0.22.04.1) ...
Setting up libavdevice58:amd64 (7:4.4.2-0ubuntu0.22.04.1) ...
Setting up ffmpeg (7:4.4.2-0ubuntu0.22.04.1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
Processing triggers for libgdk-pixbuf-2.0-0:amd64 (2.42.8+dfsg-1ubuntu0.5) ...


In [5]:
# Optional: install libjpeg-turbo Python binding for fastest JPEG encoding
# The server auto-detects it; falls back to cv2 or PIL if not installed.
# On A100 RunPod this gives ~0.5ms/frame vs ~3ms with PIL.
!pip install -q PyTurboJPEG 2>&1 | tail -3
try:
    from turbojpeg import TurboJPEG
    print("✅ TurboJPEG available — fastest JPEG encoding enabled")
except ImportError:
    print("ℹ️  TurboJPEG not available — will use cv2 or PIL (still functional)")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ TurboJPEG available — fastest JPEG encoding enabled


In [6]:
# TensorRT (for Ditto)
!pip install -q filetype cython huggingface_hub 
!pip install "turbojpeg<2"
!pip install -q cuda-python==12.4.0
!python -c "from cuda import cuda, cudart, nvrtc; print('✅ cuda-python OK')"
!pip install -q tensorrt==8.6.1 --extra-index-url https://pypi.nvidia.com
!python -c "import tensorrt as trt; print('✅ TRT', trt.__version__)"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.2/565.2 kB 38.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ cuda-python OK

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ TRT 8.6.1


In [7]:
# ── Fix: upgrade typing_extensions so FastAPI can import 'Sentinel' ──────────
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
    return r.returncode

print('Step 1/3 — Upgrading typing_extensions to >=4.6.0 ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'typing_extensions>=4.6.0', '--upgrade'])
print('✅ done' if rc == 0 else '❌ failed')

print('Step 2/3 — Reinstalling FastAPI with compatible version ...')
rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'fastapi>=0.100.0', '--upgrade', '--force-reinstall', '--no-deps'])
print('✅ done' if rc == 0 else '❌ failed')

rc = run([sys.executable, '-m', 'pip', 'install', '-q',
          'starlette>=0.27.0', '--upgrade'])

print('Step 3/3 — Verifying fix ...')
try:
    import importlib, sys as _sys
    for mod in list(_sys.modules.keys()):
        if 'fastapi' in mod or 'typing_extensions' in mod or 'starlette' in mod:
            del _sys.modules[mod]
    import fastapi
    print(f'✅ FastAPI {fastapi.__version__} imports successfully!')
except Exception as e:
    print(f'❌ Still failing: {e}')
    print('   → Try restarting the kernel and re-running this cell.')

Step 1/3 — Upgrading typing_extensions to >=4.6.0 ...
✅ done
Step 2/3 — Reinstalling FastAPI with compatible version ...
✅ done
Step 3/3 — Verifying fix ...
✅ FastAPI 0.135.3 imports successfully!


In [8]:
# ── ONNX Runtime GPU (for Bridge ONNX model) ──────────────────────────────────
# onnxruntime-gpu must match the CUDA version on the RunPod instance.
# CUDA 12.x → onnxruntime-gpu >= 1.17
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "onnxruntime-gpu>=1.17.0",
     "--extra-index-url", "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode != 0:
    print("⚠️  onnxruntime-gpu install failed — trying CPU fallback")
    !pip install -q onnxruntime

# Verify
import onnxruntime as ort
print(f"✅ ONNX Runtime {ort.__version__}")
print(f"   Providers: {ort.get_available_providers()}")


✅ ONNX Runtime 1.24.4
   Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


---
## ✅ Cell 2 — Verify Environment

In [9]:
import torch, shutil, sys

print("=" * 58)
print("  Environment Check")
print("=" * 58)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA ok : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

for pkg in ['fastapi','uvicorn','onnxruntime']:
    try:
        m = __import__(pkg)
        print(f"  {pkg:<14}: {m.__version__}")
    except ImportError as e:
        print(f"  {pkg:<14}: ❌ {e}")

print(f"  FFmpeg  : {'✅' if shutil.which('ffmpeg') else '❌ NOT FOUND'}")
print("=" * 58)

  Environment Check
  Python  : 3.11.10
  PyTorch : 2.5.1+cu124
  CUDA ok : True
  GPU     : NVIDIA A100-SXM4-80GB
  VRAM    : 85.1 GB
  fastapi       : 0.135.3
  uvicorn       : 0.44.0
  onnxruntime   : 1.24.4
  FFmpeg  : ✅


---
## ⚙️ Cell 3 — Download Checkpoints

In [10]:
from huggingface_hub import snapshot_download, hf_hub_download
import os

PROJECT_ROOT = os.path.abspath(os.getcwd())
CKPT_DIR     = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Bridge checkpoint (.pt) ───────────────────────────────────────────────────
print("📥 Downloading bridge_best.pt...")
snapshot_download(
    repo_id="Darknsu/librispeech-full-dataset-model",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
    allow_patterns=["bridge_best.pt"],
)
print("✅ bridge_best.pt downloaded.")

# ── Bridge checkpoint (.onnx) ─────────────────────────────────────────────────
print("\n📥 Downloading bridge_best.onnx...")
snapshot_download(
    repo_id="Darknsu/bridge_best_onnx",
    repo_type="dataset",
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
)
print("✅ bridge_best.onnx downloaded.")

# ── Ditto TRT models ──────────────────────────────────────────────────────────
print("\n📥 Downloading Ditto TRT models (may take a few minutes)...")
snapshot_download(
    repo_id="digital-avatar/ditto-talkinghead",
    repo_type="model",
    local_dir=os.path.join(PROJECT_ROOT, "ditto-inference", "checkpoints"),
    local_dir_use_symlinks=False,
)
print("✅ Ditto models downloaded.")

📥 Downloading bridge_best.pt...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

bridge_best.pt:   0%|          | 0.00/251M [00:00<?, ?B/s]

✅ bridge_best.pt downloaded.

📥 Downloading bridge_best.onnx...


...

ditto_trt_Ampere_Plus/lmdm_v0.4_hubert_f(…):   0%|          | 0.00/195M [00:00<?, ?B/s]

ditto_trt_Ampere_Plus/motion_extractor_f(…):   0%|          | 0.00/120M [00:00<?, ?B/s]

ditto_trt_Ampere_Plus/stitch_network_fp1(…):   0%|          | 0.00/381k [00:00<?, ?B/s]

ditto_trt_Ampere_Plus/warp_network_fp16.(…):   0%|          | 0.00/107M [00:00<?, ?B/s]

✅ Ditto models downloaded.


---
## 🔧 Cell 4 — Configure Server

**Edit paths here.** ONNX bridge is preferred by default — set `USE_ONNX_BRIDGE = False` to force the `.pt` streaming mode.

In [11]:
import os, torch

PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

# ════════════════════════════════════════════════════
# 🔧 Server settings
# ════════════════════════════════════════════════════
SERVER_HOST = "0.0.0.0"
SERVER_PORT = 7860

# ════════════════════════════════════════════════════
# 🔑 Moshi
# ════════════════════════════════════════════════════
MOSHI_HF_REPO   = "kyutai/moshiko-pytorch-bf16"
MOSHI_WEIGHT    = None   # local path to skip HF download
MIMI_WEIGHT     = None
MOSHI_TOKENIZER = None

# ════════════════════════════════════════════════════
# 🔑 Bridge — ONNX is preferred for lowest latency
# ════════════════════════════════════════════════════
BRIDGE_ONNX   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.onnx")  # preferred
BRIDGE_CKPT   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")    # fallback
BRIDGE_CONFIG = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")
BRIDGE_CHUNK  = 1         # 1 = process every token immediately (ONNX mode)
USE_ONNX_BRIDGE = True    # Set False to force .pt KV-cache streaming mode

# ════════════════════════════════════════════════════
# 🔑 Ditto
# ════════════════════════════════════════════════════
DITTO_DATA_ROOT = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_trt_Ampere_Plus")
DITTO_CFG_PKL   = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_cfg",
                                "v0.4_hubert_cfg_trt.pkl")
DITTO_EMO             = 4    # emotion index 0-7
DITTO_SAMPLING_STEPS  = 10   # ↓ reduces diffusion latency (try 10–20 for speed)
DITTO_JPEG_QUALITY    = 75   # ↓ smaller JPEG = faster transfer

# ════════════════════════════════════════════════════
# ✅ Validate paths
# ════════════════════════════════════════════════════
print("\n📋 Path Check")
print("─" * 60)
errors = []
checks = [
    ("Bridge .onnx",     BRIDGE_ONNX),
    ("Bridge .pt",       BRIDGE_CKPT),
    ("Bridge config",    BRIDGE_CONFIG),
    ("Ditto data root",  DITTO_DATA_ROOT),
    ("Ditto cfg pkl",    DITTO_CFG_PKL),
]
for label, path in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'}  {label:<22} {path}")
    if not exists and label != "Bridge .onnx":
        errors.append(label)

# Check ONNX Runtime
try:
    import onnxruntime as ort
    providers = ort.get_available_providers()
    gpu_ok    = "CUDAExecutionProvider" in providers
    print(f"  {'✅' if gpu_ok else '⚠️ '}  ONNX Runtime GPU     {'available' if gpu_ok else 'NOT available (CPU only)'}")
except ImportError:
    print(f"  ❌  onnxruntime          NOT installed — run onnxruntime install cell")
    errors.append("onnxruntime")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
onnx_exists = os.path.isfile(BRIDGE_ONNX)
bridge_mode = "ONNX 🚀" if (USE_ONNX_BRIDGE and onnx_exists) else ".pt streaming"
print(f"\n  Device      : {DEVICE}")
print(f"  Bridge mode : {bridge_mode}")
print(f"  DITTO steps : {DITTO_SAMPLING_STEPS} (lower=faster)")

if errors:
    print(f"\n⚠️  {len(errors)} issue(s): {errors}")
else:
    print("\n✅ All checks passed — ready to launch!")

📁 Project root: /workspace/moshi-ditto-streaming-debug-v11

📋 Path Check
────────────────────────────────────────────────────────────
  ✅  Bridge .onnx           /workspace/moshi-ditto-streaming-debug-v11/checkpoints/bridge_best.onnx
  ✅  Bridge .pt             /workspace/moshi-ditto-streaming-debug-v11/checkpoints/bridge_best.pt
  ✅  Bridge config          /workspace/moshi-ditto-streaming-debug-v11/bridge_module/config.yaml
  ✅  Ditto data root        /workspace/moshi-ditto-streaming-debug-v11/ditto-inference/checkpoints/ditto_trt_Ampere_Plus
  ✅  Ditto cfg pkl          /workspace/moshi-ditto-streaming-debug-v11/ditto-inference/checkpoints/ditto_cfg/v0.4_hubert_cfg_trt.pkl
  ✅  ONNX Runtime GPU     available

  Device      : cuda
  Bridge mode : ONNX 🚀
  DITTO steps : 10 (lower=faster)

✅ All checks passed — ready to launch!


---
## 🚀 Cell 5a — Kill Any Existing Server

In [12]:
import os, signal, time

def get_pids_on_port(port):
    hex_port = format(port, '04X')
    inodes = set()
    for path in ["/proc/net/tcp", "/proc/net/tcp6"]:
        try:
            with open(path) as f:
                for line in f.readlines()[1:]:
                    parts = line.split()
                    if len(parts) > 9 and parts[1].split(":")[1] == hex_port:
                        inodes.add(parts[9])
        except FileNotFoundError:
            pass
    pids = []
    for pid in os.listdir("/proc"):
        if not pid.isdigit(): continue
        try:
            for fd in os.listdir(f"/proc/{pid}/fd"):
                try:
                    link = os.readlink(f"/proc/{pid}/fd/{fd}")
                    if "socket:[" in link and link.split("[")[1].rstrip("]") in inodes:
                        pids.append(int(pid)); break
                except OSError: pass
        except OSError: pass
    return pids

print("🔪 Killing any existing server processes...")
killed = []
for pid in os.listdir("/proc"):
    if not pid.isdigit(): continue
    try:
        with open(f"/proc/{pid}/cmdline") as f:
            if "streaming_server.py" in f.read():
                os.kill(int(pid), signal.SIGKILL); killed.append(pid)
    except (OSError, PermissionError): pass

if killed: print(f"   💀 Killed PIDs: {', '.join(killed)}")
time.sleep(1)

remaining = get_pids_on_port(SERVER_PORT)
for pid in remaining:
    try: os.kill(pid, signal.SIGKILL); print(f"   💀 Killed port-holder PID {pid}")
    except: pass
if remaining: time.sleep(1)

if get_pids_on_port(SERVER_PORT):
    print(f"⚠️  Port {SERVER_PORT} still busy!")
else:
    print(f"✅ Port {SERVER_PORT} is free — safe to launch.")

🔪 Killing any existing server processes...
✅ Port 7860 is free — safe to launch.


---
## 🚀 Cell 5b — Launch Streaming Server

All three models load at startup (~60–120s). Once **All models loaded** appears, open the URL.

In [ ]:
import os, sys, subprocess, threading, time
from IPython.display import display, HTML

PROJECT_ROOT = os.path.abspath(os.getcwd())

# ── Environment overrides ──────────────────────────────────────────────────────
env_overrides = {
    "MOSHI_HF_REPO":        MOSHI_HF_REPO,
    "BRIDGE_ONNX":          BRIDGE_ONNX,
    "BRIDGE_CKPT":          BRIDGE_CKPT,
    "BRIDGE_CONFIG":        BRIDGE_CONFIG,
    "BRIDGE_CHUNK":         str(BRIDGE_CHUNK),
    "USE_ONNX_BRIDGE":      "1" if USE_ONNX_BRIDGE else "0",
    "DITTO_DATA_ROOT":      DITTO_DATA_ROOT,
    "DITTO_CFG_PKL":        DITTO_CFG_PKL,
    "DITTO_EMO":            str(DITTO_EMO),
    "DITTO_SAMPLING_STEPS": str(DITTO_SAMPLING_STEPS),
    "DITTO_JPEG_QUALITY":   str(DITTO_JPEG_QUALITY),
}
if MOSHI_WEIGHT:    env_overrides["MOSHI_WEIGHT"]    = MOSHI_WEIGHT
if MIMI_WEIGHT:     env_overrides["MIMI_WEIGHT"]     = MIMI_WEIGHT
if MOSHI_TOKENIZER: env_overrides["MOSHI_TOKENIZER"] = MOSHI_TOKENIZER

env = {**os.environ, **env_overrides}

# ── Public URL ────────────────────────────────────────────────────────────────
hostname   = os.environ.get("RUNPOD_POD_HOSTNAME",
             os.environ.get("RUNPOD_POD_ID", "localhost"))
PUBLIC_URL = (f"https://{hostname}-{SERVER_PORT}.proxy.runpod.net"
              if hostname != "localhost"
              else f"http://localhost:{SERVER_PORT}")

# ── Launch server ─────────────────────────────────────────────────────────────
server_script = os.path.join(PROJECT_ROOT, "streaming_server.py")
cmd = [sys.executable, "-u", server_script,
       "--host", SERVER_HOST, "--port", str(SERVER_PORT)]

print(f"🚀 Launching server  →  {PUBLIC_URL}\n")

proc = subprocess.Popen(
    cmd, cwd=PROJECT_ROOT, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

_ready = threading.Event()

def _stream_logs():
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        if "All models loaded" in line or "Application startup complete" in line:
            _ready.set()
    print("\n⚠️  Server process exited.", flush=True)

threading.Thread(target=_stream_logs, daemon=True).start()

print("⏳ Waiting for models to load (1–2 min)…\n")
if _ready.wait(timeout=360):
    print("\n" + "═" * 62)
    print(f"  ✅  Server ready!")
    print(f"  🌐  {PUBLIC_URL}")
    print("═" * 62)
    display(HTML(
        f'<a href="{PUBLIC_URL}" target="_blank" '
        f'style="font-size:1.4em;font-weight:bold;color:#4f8ef7;">'
        f'▶ Open LiveAvatar ({PUBLIC_URL})</a>'
    ))
else:
    print("❌ Timed out — check logs above.")

# Keep cell alive so logs keep streaming
print("\n📡 Live logs (⬛ Interrupt to stop):\n")
try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); proc.wait()
    print("\n🛑 Server stopped.")

🚀 Launching server  →  https://7kjt0vayf0kx7i-64411aa0-7860.proxy.runpod.net

⏳ Waiting for models to load (1–2 min)…

In file included from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1913,
                 from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /root/.pyxbld/temp.linux-x86_64-cpython-311/workspace/moshi-ditto-streaming-debug-v11/pipeline/../ditto-inference/core/utils/blend/blend.c:1157:
/usr/local/lib/python3.11/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
/workspace/moshi-ditto-streaming-debug-v11/streaming_server.py:512: DeprecationWarning:


📡 Live logs (⬛ Interrupt to stop):

INFO:     100.64.1.33:32846 - "GET / HTTP/1.1" 200 OK
INFO:     100.64.1.32:51544 - "GET /favicon.ico HTTP/1.1" 404 Not Found
01:24:37  INFO      streaming_server  [upload_image] Session 1561da24 → /workspace/moshi-ditto-streaming-debug-v11/_uploads/1561da24.jpeg
INFO:     100.64.1.34:54556 - "POST /upload_image HTTP/1.1" 200 OK
INFO:     100.64.1.34:59518 - "WebSocket /ws/1561da24" [accepted]
01:24:40  INFO      streaming_server  [WS] Client connected — session 1561da24
01:24:40  INFO      latency_tracker  [01:24:40.709] [EVENT] Session started  image=/workspace/moshi-ditto-streaming-debug-v11/_uploads/1561da24.jpeg  ditto_steps=10  bridge_chunk=1  jpeg_q=75
01:24:40  INFO      pipeline.ditto_stream_adapter  [DittoStreamAdapter] Loading StreamSDK from /workspace/moshi-ditto-streaming-debug-v11/ditto-inference/checkpoints/ditto_trt_Ampere_Plus …
01:24:43  INFO      pipeline.ditto_stream_adapter  [DittoStreamAdapter] StreamSDK loaded.
===============

---
## 🛑 Cell 6 — Stop Server

In [ ]:
try:
    proc.terminate(); proc.wait(timeout=10)
    print("✅ Server stopped.")
except NameError:
    print("Server was not running.")
except Exception as e:
    print(f"Error: {e}"); proc.kill()

---
## 🔬 Cell 7 — ONNX Bridge Smoke Test

Run this cell independently to verify the ONNX bridge works before launching the full server.

In [ ]:
import sys, os, time, torch, numpy as np

PROJECT_ROOT = os.path.abspath(os.getcwd())
BRIDGE_DIR   = os.path.join(PROJECT_ROOT, "bridge_module")
if BRIDGE_DIR not in sys.path: sys.path.insert(0, BRIDGE_DIR)

from inference import OnnxBridgeInference, BridgeInference

ONNX_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.onnx")
PT_PATH   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")
CFG_PATH  = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")

# ── ONNX test ─────────────────────────────────────────────────────────────────
if os.path.isfile(ONNX_PATH):
    print("🧪 Testing ONNX bridge...")
    bridge = OnnxBridgeInference(ONNX_PATH, CFG_PATH, device="cuda")

    # Dummy tokens: (1, 1, 8) — 1 batch, 1 time step, 8 codebooks
    dummy = torch.zeros(1, 1, 8, dtype=torch.int64)

    # Warmup
    for _ in range(3): bridge(dummy)

    # Latency test
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        out = bridge(dummy)
        times.append((time.perf_counter() - t0) * 1000)

    print(f"  Output shape : {out.shape}")
    print(f"  Median latency: {sorted(times)[25]:.2f} ms  (per 1 token)")
    print(f"  P95 latency   : {sorted(times)[47]:.2f} ms")
    print("  ✅ ONNX bridge OK")
else:
    print(f"⚠️  bridge_best.onnx not found at {ONNX_PATH}")
    print("   Run Cell 3 to download it, or set USE_ONNX_BRIDGE = False to use .pt mode.")

---
## 📊 Cell 8 — GPU Memory Check

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(0) / 1e9
    reserv = torch.cuda.memory_reserved(0)  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔬 GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Allocated : {alloc:.2f} GB")
    print(f"  Reserved  : {reserv:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserv:.2f} GB")
else:
    print("CUDA not available.")

---
<div style="background:#080c14;padding:20px;border-radius:10px;text-align:center;border:1px solid rgba(79,142,247,.2);">
  <h3 style="color:#4f8ef7;margin:0;">⚡ Quick Reference</h3>
  <table style="color:#64748b;margin:14px auto;border-collapse:collapse;font-family:sans-serif;font-size:13px;">
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Install</td>
        <td style="padding:5px 20px;">→ Cell 1 (once per session)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Download</td>
        <td style="padding:5px 20px;">→ Cell 3 (.pt + .onnx + Ditto)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Configure</td>
        <td style="padding:5px 20px;">→ Cell 4 (set paths & options)</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Kill old server</td>
        <td style="padding:5px 20px;">→ Cell 5a</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Launch server</td>
        <td style="padding:5px 20px;">→ Cell 5b 🚀</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Stop server</td>
        <td style="padding:5px 20px;">→ Cell 6 🛑</td></tr>
    <tr><td style="padding:5px 20px;text-align:right;font-weight:bold;color:#94a3b8;">Test ONNX</td>
        <td style="padding:5px 20px;">→ Cell 7 🧪</td></tr>
  </table>
  <p style="color:#4f8ef7;font-size:12px;margin-top:10px;">
    Browser: Upload portrait → Connect → Start Speaking → See live talking head + hear audio
  </p>
</div>